# Pocket-finder box → oriented docking search box

Manual verification notebook for [DDOS-7327](https://deeporigin.atlassian.net/browse/DDOS-7327).

Run against **dev** (`DO_ENV=dev`, `deeporigin login`). Requires pocket-finder
**1.6.19+** with nested `box` output (PCA OBB sizes + `rotation_deg`).

Flow:
1. Sync a protein and run pocket-finder
2. Inspect `pocket.box` from the live run
3. Build `Docking(...)`, preview the oriented search box with `show_box()`
4. Run docking, fetch poses, and overlay them with `show_box(poses=...)`

In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from deeporigin.drug_discovery import (
    BRD_DATA_DIR,
    Docking,
    Ligand,
    PocketFinder,
    Protein,
)
from deeporigin.platform.client import DeepOriginClient

client = DeepOriginClient.from_disk()
client.base_url

## Load and sync protein

In [ ]:
protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
protein.remove_water()
protein.sync(client=client)
protein.id

## Run pocket-finder (live dev)

Pocket-finder emits parent lab-frame `box_size_*` plus nested `box` with
PCA-aligned OBB sizes and `rotation_deg`.

In [ ]:
pf = PocketFinder(protein, pocket_count=1, client=client)
pockets = pf.run()
pocket = pockets[0]
pocket

## Inspect nested `box`

Confirm pocket-finder returned oriented box metadata on the `Pocket` object.

In [ ]:
assert pocket.box is not None, "Expected pocket.box from pocket-finder 1.6.19+"
assert "box" not in (pocket.props or {}), "box should be a first-class Pocket.box field"

print("Lab-frame AABB (parent):")
print(
    f"  box_size_x/y/z = {pocket.box_size_x}, {pocket.box_size_y}, {pocket.box_size_z}"
)
print("PCA-aligned OBB (nested box):")
print(
    f"  box_size_x/y/z = {pocket.box.box_size_x}, {pocket.box.box_size_y}, {pocket.box.box_size_z}"
)
print(f"  rotation_deg   = {pocket.box.rotation_deg}")
assert len(pocket.box.rotation_deg) == 3

## Load ligand

In [ ]:
ligand = Ligand.from_sdf(BRD_DATA_DIR / "brd-2.sdf")
ligand.sync(client=client)
ligand.id

## Build docking from pocket

In [ ]:
docking = Docking(
    protein=protein,
    pocket=pocket,
    ligand=ligand,
    client=client,
)
docking

## Show oriented search box

Static `show_box()` should render the wireframe with `pocket.box.rotation_deg`
already applied — verify visually that the box aligns with the pocket geometry.

In [ ]:
docking.show_box(interactive=True)

## Sanity: tool payload matches viz

In [ ]:
params, _ = docking._build_tool_inputs()
pocket_in = params["pocket"]
print("Tool pocket params:")
print(pocket_in)
assert pocket_in.get("rotation_deg") == docking._effective_docking_rotation_deg()
assert pocket_in["box_size_x"] == pocket.box.box_size_x

In [ ]:
docking.rotation_deg

## Run docking

Dock the ligand into the pocket using the oriented search box from pocket-finder.
This blocks until the run completes on dev.

In [ ]:
docking.run(quote=True)
docking.estimate


In [ ]:
poses = docking.run()

## Fetch poses

Reload docked poses from the platform (same result as the synchronous `run()` return value).

In [ ]:
poses = docking.get_poses()
len(poses), poses[0].pose_score, poses[0].binding_energy

## Show poses with oriented search box

Overlay docked poses on the protein with the pocket-finder oriented wireframe.

In [ ]:
poses.download(client=client)
docking.show_box(poses=poses[0])

In [ ]:
poses.to_dataframe()